# Week 1 · Day 4 — Intro to pandas (DataFrames)

*A spreadsheet you can automate.*

**By the end you'll have shipped:** load a **matters table**, filter it, and produce a **billing summary by practice area** — in about five lines.

> Core Path = everything unmarked. `Go Deeper 🔧` = optional.

### 📋 Lesson card

| | |
|---|---|
| **Module** | M1 · Python foundations (Week 1) |
| **Prerequisites** | Days 1–3 (variables, lists/dicts, functions) |
| **Est. time** | ~30 min |
| **Capstone tie-in** | *Matter Intelligence* — the table we filter/aggregate is what later lands in Snowflake |
| **Difficulty** | Core (+ optional Go Deeper) |

### 🎯 Learning objectives

- Explain what a **DataFrame** is (a smart, programmable spreadsheet).
- **Create** one from a list of dicts and **read** one from a CSV file.
- **Select** columns and **filter** rows with a condition.
- **Sort** and **aggregate** with `groupby` — and see how these map to **SQL** you'll teach next.

### ⚖️ Why it matters

Your team already thinks in spreadsheets of matters. **pandas** gives you that spreadsheet *in code*, so you can filter 10,000 matters as easily as 10, repeat it every morning, and feed the results to Claude or Snowflake. Crucially, the four moves you'll learn today — **select, filter, sort, group** — are the exact moves of **SQL** (`SELECT`, `WHERE`, `ORDER BY`, `GROUP BY`). Learn them here in Python and SQL will feel familiar when you teach it for Snowflake.

### ⚙️ Setup

This cell imports pandas and makes sure a sample `matters.csv` exists **right next to this notebook**, so the lesson runs offline no matter where you launched Jupyter. (It uses the shared `Training/data/matters.csv` if it can find it, otherwise it writes a local copy.)

In [ ]:
import os
import pandas as pd

# The synthetic docket we'll use all lesson (same shape as Day 2's list-of-dicts).
SAMPLE = [
    {"matter_id":"M-1001","client":"Acme Corp","practice_area":"Contracts","status":"Active","amount_billed":18500.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
    {"matter_id":"M-1002","client":"Brightline LLC","practice_area":"Litigation","status":"Active","amount_billed":42750.50,"is_privileged":True,"lead_attorney":"S. Patel"},
    {"matter_id":"M-1003","client":"Cedar Holdings","practice_area":"M&A","status":"Closed","amount_billed":131200.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
    {"matter_id":"M-1004","client":"Delta Foods","practice_area":"Employment","status":"Active","amount_billed":7300.00,"is_privileged":False,"lead_attorney":"J. Okafor"},
    {"matter_id":"M-1005","client":"Evergreen Inc","practice_area":"Contracts","status":"On Hold","amount_billed":2450.00,"is_privileged":False,"lead_attorney":"S. Patel"},
    {"matter_id":"M-1006","client":"Foster & Sons","practice_area":"Litigation","status":"Active","amount_billed":58900.75,"is_privileged":True,"lead_attorney":"J. Okafor"},
    {"matter_id":"M-1007","client":"Granite Partners","practice_area":"M&A","status":"Active","amount_billed":96400.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
    {"matter_id":"M-1009","client":"Ivory Systems","practice_area":"IP","status":"Active","amount_billed":33150.25,"is_privileged":True,"lead_attorney":"L. Romano"},
    {"matter_id":"M-1011","client":"Keystone Bank","practice_area":"Litigation","status":"On Hold","amount_billed":71200.00,"is_privileged":True,"lead_attorney":"J. Okafor"},
    {"matter_id":"M-1013","client":"Meridian Group","practice_area":"M&A","status":"Active","amount_billed":112500.00,"is_privileged":True,"lead_attorney":"R. Nguyen"},
]

CSV_PATH = "matters.csv"                     # local to this notebook
SHARED = os.path.join("..", "..", "data", "matters.csv")  # Training/data/matters.csv
if os.path.exists(SHARED):
    CSV_PATH = SHARED
elif not os.path.exists(CSV_PATH):
    pd.DataFrame(SAMPLE).to_csv(CSV_PATH, index=False)   # write a local copy so we can read it

print("pandas", pd.__version__, "— using CSV:", CSV_PATH)

### 1 · A DataFrame is a smart spreadsheet

A **DataFrame** is a table: rows (matters) and named columns (fields). We can build one straight from the list-of-dicts you already understand from Day 2.

In [ ]:
df = pd.DataFrame(SAMPLE)

df.head()          # first 5 rows, nicely rendered (run this cell to see the table)

In [ ]:
print("shape (rows, cols):", df.shape)
print("columns:", list(df.columns))
df.info()          # dtypes + non-null counts — your first data-quality glance

**What just happened:** `df.head()` shows the top rows, `df.shape` gives (rows, columns), and `df.info()` reports each column's type — pandas inferred numbers, text, and booleans for us.

### 2 · Read a table from a CSV

In real life the data lives in a file. `pd.read_csv` loads it in one line — this is how most analyses start.

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Loaded", len(df), "matters from", CSV_PATH)
df.head()

### 3 · Select columns  →  like SQL `SELECT`

Grab one column (a **Series**) with `df["col"]`, or several with a list of names. Picking columns is exactly what SQL's `SELECT` does.

In [ ]:
# one column
print(df["client"].head(3))
print("-" * 30)

# several columns  (SQL: SELECT client, practice_area, amount_billed)
df[["client", "practice_area", "amount_billed"]].head()

### 4 · Filter rows  →  like SQL `WHERE`

Write a **condition** to keep only the rows you want. `df["amount_billed"] > 50000` produces True/False for every row (remember booleans from Day 1); putting it inside `df[ ... ]` keeps the True rows.

In [ ]:
# high-value matters  (SQL: WHERE amount_billed > 50000)
high = df[df["amount_billed"] > 50000]
print(len(high), "high-value matters")
high[["matter_id", "client", "amount_billed"]]

In [ ]:
# combine conditions with & (and) / | (or) — each condition needs ( )
# active AND high-value  (SQL: WHERE status='Active' AND amount_billed > 50000)
active_high = df[(df["status"] == "Active") & (df["amount_billed"] > 50000)]
active_high[["matter_id", "client", "status", "amount_billed"]]

### 5 · Sort  →  like SQL `ORDER BY`

`sort_values` orders the table by a column.

In [ ]:
# biggest matters first  (SQL: ORDER BY amount_billed DESC)
df.sort_values("amount_billed", ascending=False)[["matter_id", "client", "amount_billed"]].head()

### 6 · Aggregate with groupby  →  like SQL `GROUP BY`

The big one. **Group** rows by a category, then compute a number per group — subtotal billings by practice area, count matters per attorney, and so on. This is `GROUP BY` in SQL, and it's where data work gets powerful.

In [ ]:
# total billings per practice area  (SQL: SELECT practice_area, SUM(amount_billed) ... GROUP BY practice_area)
by_area = df.groupby("practice_area")["amount_billed"].sum().sort_values(ascending=False)
print(by_area)

In [ ]:
# count of matters per attorney  (SQL: SELECT lead_attorney, COUNT(*) ... GROUP BY lead_attorney)
df.groupby("lead_attorney").size().sort_values(ascending=False)

> **🔗 Bridge to SQL (what you'll teach next).** Notice the pattern — every pandas move you just made has a one-to-one SQL twin:
>
> | Goal | pandas | SQL |
> |---|---|---|
> | pick columns | `df[["a","b"]]` | `SELECT a, b` |
> | keep some rows | `df[df.x > 5]` | `WHERE x > 5` |
> | order | `df.sort_values("x")` | `ORDER BY x` |
> | subtotal by group | `df.groupby("g")["x"].sum()` | `GROUP BY g` |
>
> Same thinking, two dialects. pandas is the training wheels; Snowflake SQL is the same ride at scale.

> **`Go Deeper 🔧` — `.loc`, computed columns, `value_counts()`.** `.loc[rows, cols]` selects by label; you can add a computed column in one line; and `value_counts()` is a fast category tally.

In [ ]:
# Go Deeper (optional)
# 1) add a computed column
df["fee_band"] = df["amount_billed"].apply(lambda x: "large" if x >= 50000 else "standard")

# 2) .loc selects rows (by condition) and columns (by name) together
print(df.loc[df["fee_band"] == "large", ["matter_id", "client", "amount_billed"]])

# 3) quick category tally
print("\nMatters by status:")
print(df["status"].value_counts())

> **`Common pitfalls ⚠️`**
>
> - Combine filters with `&` / `|` (not `and`/`or`), and wrap **each** condition in parentheses.
> - `=` assigns; `==` compares — filters use `==`.
> - Column names are case-sensitive and must match exactly (`"amount_billed"`).
> - A `SettingWithCopyWarning` usually means you filtered then assigned — create with `.copy()` if you plan to edit a subset.

### ✍️ Your turn

In [ ]:
# Using df:
# TODO 1: show only matters where status == 'Active'
# TODO 2: from those, select just matter_id, client, amount_billed
# TODO 3: compute the AVERAGE amount_billed per practice_area (hint: .mean())
# TODO 4 (stretch): how many Active matters does each lead_attorney have?

# your code here


<details><summary>✅ Show solution</summary>

```python
# 1 & 2
active = df[df["status"] == "Active"]
print(active[["matter_id", "client", "amount_billed"]])

# 3
print(df.groupby("practice_area")["amount_billed"].mean())

# 4
print(active.groupby("lead_attorney").size())
```
</details>

### 🚀 Build the artifact — a billing summary by practice area

The full pipeline in a handful of lines: **read → filter → group → save**. This is a real, repeatable report — the kind of thing you'd later schedule to run every morning.

In [ ]:
# 1. read
df = pd.read_csv(CSV_PATH)

# 2. filter to open work (not Closed)
open_df = df[df["status"] != "Closed"]

# 3. group: total & average billings by practice area, plus a matter count
summary = (
    open_df.groupby("practice_area")
           .agg(matters=("matter_id", "count"),
                total_billed=("amount_billed", "sum"),
                avg_billed=("amount_billed", "mean"))
           .sort_values("total_billed", ascending=False)
           .round(2)
)
print(summary)

# 4. save the report for sharing
summary.to_csv("billing_summary_by_area.csv")
print("\n✅ Shipped: billing_summary_by_area.csv")

### 📝 Recap — what you shipped

- A **DataFrame** is a programmable spreadsheet; build it from dicts or `pd.read_csv`.
- **Select** columns, **filter** rows (with `&`/`|`), **sort**, and **group** to aggregate.
- Those four moves map directly to SQL `SELECT` / `WHERE` / `ORDER BY` / `GROUP BY`.
- **Artifact:** a saved billing-summary-by-practice-area report.

### 🧠 Check your understanding

1. Which pandas operation is the twin of SQL's `WHERE`?
2. Why do combined filters need `&`/`|` and parentheses instead of `and`/`or`?
3. What does `df.groupby("practice_area")["amount_billed"].sum()` give you?

<details><summary>Answers</summary>

1. **Filtering** rows with a condition: `df[df["x"] > 5]`.
2. pandas evaluates the condition across the whole column at once (element-wise); `&`/`|` do that, and parentheses fix the order of operations.
3. The **total billed for each practice area** — one subtotal per group.
</details>

### ➡️ Next up

That wraps **Week 1 — Python foundations**: variables, collections & loops, functions, and pandas. You can now read, shape, and summarize legal data in code.

**Where this goes next:**
- **SQL for Snowflake** — you just met its four core verbs in pandas; next we write them as real SQL against a matters table (with a local SQLite fallback so it runs offline).
- **Building with Claude** — feed a filtered DataFrame of matters into an LLM to summarize or classify them (the *Matter Intelligence* capstone).

### 📖 Reference & glossary

| Term | Plain meaning | SQL twin |
|---|---|---|
| DataFrame | a programmable table | a table |
| Series | one column | one column |
| `df[[...]]` | pick columns | `SELECT` |
| `df[df.x > n]` | keep matching rows | `WHERE` |
| `sort_values` | order rows | `ORDER BY` |
| `groupby(...).agg(...)` | subtotal by category | `GROUP BY` |

**Official docs:** [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) · [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [`groupby`](https://pandas.pydata.org/docs/user_guide/groupby.html)